In [34]:
from pyspark.sql import SparkSession

In [35]:
import os

In [36]:
os.environ["JAVA_HOME"] = r"C:\Users\alexandre.karneff\Java\jdk-17"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

In [37]:
spark = SparkSession.builder.appName("Practice").getOrCreate()
spark

In [38]:
df_pyspark = spark.read.csv("data/car_data.csv",inferSchema=True, header=True)

In [39]:
df_pyspark.show(5)

+------+------------+-----+-------+--------+------+--------+
|buying|maintainence|doors|persons|lug_boot|safety|car_type|
+------+------------+-----+-------+--------+------+--------+
| vhigh|       vhigh|    2|      2|   small|   low|   unacc|
| vhigh|       vhigh|    2|      2|   small|   med|   unacc|
| vhigh|       vhigh|    2|      2|   small|  high|   unacc|
| vhigh|       vhigh|    2|      2|     med|   low|   unacc|
| vhigh|       vhigh|    2|      2|     med|   med|   unacc|
+------+------------+-----+-------+--------+------+--------+
only showing top 5 rows


In [40]:
df_pyspark.printSchema()

root
 |-- buying: string (nullable = true)
 |-- maintainence: string (nullable = true)
 |-- doors: string (nullable = true)
 |-- persons: string (nullable = true)
 |-- lug_boot: string (nullable = true)
 |-- safety: string (nullable = true)
 |-- car_type: string (nullable = true)



In [41]:
from pyspark.ml.feature import StringIndexer

In [42]:
categoricalColumns = ["buying","maintainence","doors","persons","lug_boot","safety","car_type"]

In [43]:
l = []

In [44]:
for categoricalCol in categoricalColumns:
    stringIndexer = StringIndexer(inputCol = categoricalCol, outputCol = categoricalCol+"_encoded").fit(df_pyspark)
    df_pyspark = stringIndexer.transform(df_pyspark)
    df_pyspark = df_pyspark.withColumn(categoricalCol+"_encoded", df_pyspark[categoricalCol+"_encoded"].cast('int'))

In [45]:
encoded_df =  df_pyspark.select("buying_encoded","doors_encoded","maintainence_encoded","persons_encoded","lug_boot_encoded","safety_encoded","car_type_encoded")

In [46]:
encoded_df.show(5)

+--------------+-------------+--------------------+---------------+----------------+--------------+----------------+
|buying_encoded|doors_encoded|maintainence_encoded|persons_encoded|lug_boot_encoded|safety_encoded|car_type_encoded|
+--------------+-------------+--------------------+---------------+----------------+--------------+----------------+
|             3|            0|                   3|              0|               2|             1|               0|
|             3|            0|                   3|              0|               2|             2|               0|
|             3|            0|                   3|              0|               2|             0|               0|
|             3|            0|                   3|              0|               1|             1|               0|
|             3|            0|                   3|              0|               1|             2|               0|
+--------------+-------------+--------------------+-------------

In [47]:
from pyspark.ml.feature import VectorAssembler

In [48]:
featureAssembler = VectorAssembler(inputCols=["buying_encoded","doors_encoded","maintainence_encoded","persons_encoded","lug_boot_encoded","safety_encoded"],outputCol="features")

In [49]:
output = featureAssembler.transform(encoded_df)

In [50]:
output.select("features","car_type_encoded").show(5)

+--------------------+----------------+
|            features|car_type_encoded|
+--------------------+----------------+
|[3.0,0.0,3.0,0.0,...|               0|
|[3.0,0.0,3.0,0.0,...|               0|
|[3.0,0.0,3.0,0.0,...|               0|
|[3.0,0.0,3.0,0.0,...|               0|
|[3.0,0.0,3.0,0.0,...|               0|
+--------------------+----------------+
only showing top 5 rows


In [51]:
train, test = output.randomSplit([0.8, 0.2], seed=17)

In [52]:
print("Size of training data: ",train.count())
print("Size of testing date: ",test.count())

Size of training data:  1377
Size of testing date:  351


Logistic Regression

In [53]:
from pyspark.ml.classification import LogisticRegression

In [54]:
lr = LogisticRegression(featuresCol = 'features', labelCol = 'car_type_encoded', maxIter=10)

In [55]:
lrModel = lr.fit(train)

In [56]:
predictions = lrModel.transform(test)

In [57]:
predictions.show(5)

+--------------+-------------+--------------------+---------------+----------------+--------------+----------------+-------------------+--------------------+--------------------+----------+
|buying_encoded|doors_encoded|maintainence_encoded|persons_encoded|lug_boot_encoded|safety_encoded|car_type_encoded|           features|       rawPrediction|         probability|prediction|
+--------------+-------------+--------------------+---------------+----------------+--------------+----------------+-------------------+--------------------+--------------------+----------+
|             0|            0|                   0|              0|               0|             0|               0|          (6,[],[])|[1.99198136762971...|[0.76125517518711...|       0.0|
|             0|            0|                   0|              0|               2|             0|               0|      (6,[4],[2.0])|[3.26816128608160...|[0.90473344519298...|       0.0|
|             0|            0|                   0

In [58]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [61]:
evaluator = MulticlassClassificationEvaluator()
evaluator.setLabelCol("car_type_encoded")
evaluator.setPredictionCol("prediction")
print("Test Area under ROC: ",evaluator.evaluate(predictions))

Test Area under ROC:  0.5901985607326307


Decision Tree

In [62]:
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [63]:
#Training Model
dt = DecisionTreeClassifier(featuresCol = 'features', labelCol = 'car_type_encoded', maxDepth = 3)
dtModel = dt.fit(train)

In [64]:
#Prediction
predictions = dtModel.transform(test)

In [65]:
#Evaluating the performance
evaluator = MulticlassClassificationEvaluator()
evaluator.setLabelCol("car_type_encoded")
evaluator.setPredictionCol("prediction")
print("Test Area Under ROC: ",evaluator.evaluate(predictions))

Test Area Under ROC:  0.7398590195812419


Random Forest

In [66]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [67]:
#Training Model
rf = RandomForestClassifier(featuresCol = 'features', labelCol = 'car_type_encoded', numTrees = 500, maxDepth = 10)
rfModel = rf.fit(train)

In [68]:
#Prediction
predictions = rfModel.transform(test)

In [69]:
#Evaluating the performance
evaluator = MulticlassClassificationEvaluator()
evaluator.setLabelCol("car_type_encoded")
evaluator.setPredictionCol("prediction")
print("Test Area Under ROC: ",evaluator.evaluate(predictions))

Test Area Under ROC:  0.9830354095424948


In [70]:
spark.stop()